In [12]:
import re
from collections import Counter

import pandas as pd

In [13]:
path = "/Users/karolinegriesbach/Documents/Innkeepr/Git/evaluation-and-execution-scripts/DataChecks/check_logs_conversion_signals/"
file_name = "clockin-69b03547e150c3010de2bfb5-2026-03-11 10:34:06.027751.log"
#file_name="/Users/karolinegriesbach/Documents/Innkeepr/Git/innkeepr-analytics/junglueck-682b1d8a1de6c855565aeff2-2026-03-04 10:22:49.362045.log"

with open(f"{file_name}", "r") as f:
    content = f.read()

lines = [l for l in content.split("\n") if "message" in l]
print(f"Found {len(lines)} response lines with error messages")

In [14]:
results = []

for i, line in enumerate(lines):
    resp_match = re.search(r"response = (\{.*\})", line)
    if not resp_match:
        continue

    resp = eval(resp_match.group(1))
    data = resp.get("data", {})
    received = data.get("conversionsReceived", 0)
    uploaded = data.get("conversionsUploaded", 0)
    errors = data.get("errors", [])

    # normalize error messages: strip per-conversion index and gclid details
    msg_counter = Counter()
    for err in errors:
        msg = err.get("message", "")
        normalized = re.sub(r"\s*at \.conversions\[\d+\].*", "", msg).strip()
        msg_counter[normalized] += 1

    results.append({
        "batch": i + 1,
        "received": received,
        "uploaded": uploaded,
        "failed": received - uploaded,
        "upload_rate": round(uploaded / received * 100, 2) if received else 0,
        "total_errors": len(errors),
        "error_breakdown": dict(msg_counter.most_common()),
    })

print(f"Parsed {len(results)} batches")

In [15]:
# overview per batch
df = pd.DataFrame(results).drop(columns=["error_breakdown"])
df["upload_rate"] = df["upload_rate"].apply(lambda x: f"{x}%")
df

In [16]:
# error breakdown across all batches
all_errors = Counter()
for r in results:
    for msg, count in r["error_breakdown"].items():
        all_errors[msg] += count

df_errors = pd.DataFrame(
    [(msg, count) for msg, count in all_errors.most_common()],
    columns=["error_message", "total_count"],
)
df_errors["percentage"] = (df_errors["total_count"] / df_errors["total_count"].sum() * 100).round(2)
df_errors

In [17]:
# totals
total_received = sum(r["received"] for r in results)
total_uploaded = sum(r["uploaded"] for r in results)
total_failed = total_received - total_uploaded

print(f"Total received:  {total_received:,}")
print(f"Total uploaded:  {total_uploaded:,}")
print(f"Total failed:    {total_failed:,}")
print(f"Overall upload rate: {total_uploaded / total_received}")

In [18]:
# save summary to file
summary_lines = []
summary_lines.append(f"Log File Analysis: {file_name}")
summary_lines.append("=" * 60)
summary_lines.append("")

summary_lines.append("TOTALS")
summary_lines.append("-" * 40)
summary_lines.append(f"Total batches:       {len(results)}")
summary_lines.append(f"Total received:      {total_received:,}")
summary_lines.append(f"Total uploaded:      {total_uploaded:,}")
summary_lines.append(f"Total failed:        {total_failed:,}")
summary_lines.append(f"Overall upload rate: {total_uploaded / total_received * 100:.2f}%")
summary_lines.append("")

summary_lines.append("BATCH OVERVIEW")
summary_lines.append("-" * 40)
summary_lines.append(df.to_string(index=False))
summary_lines.append("")

summary_lines.append("ERROR BREAKDOWN (all batches)")
summary_lines.append("-" * 40)
for _, row in df_errors.iterrows():
    summary_lines.append(f"  [{row['total_count']:,}x] ({row['percentage']}%) {row['error_message']}")

summary = "\n".join(summary_lines)

output_file = f"{path}{file_name.replace('.log', '-summary.txt')}"
with open(output_file, "w") as f:
    f.write(summary)

print(f"Saved to {output_file}")